# GFTM default vs m11_new against GS2: NSTX MTM hypercube

Do GFTM at its default settings (`gftm_fw174`: `FIND_WIDTH=F`, `WIDTH=1.74`) and the `m11_new` settings reproduce the GS2 growth rate on the 1000-point NSTX MTM Latin hypercube (`beta_q_shat_ky_n1000`)? Both arms are compared to GS2 on the same cases, so the two panels share one population.

Selection: GS2 cases with $\gamma > 10^{-3}$, and GFTM's dominant mode (`argmax` of `growth_rate` over `mode`). Only cases where **both** arms return $\gamma > 0$ are scored (the paired population). Metrics use finite, strictly positive pairs: RMSE and bias raw in $c_s/a$, Pearson $r$ on $\log_{10}\gamma$. Cases are matched by `sample_name`, never by position.

## Imports and settings

Set `GK_DATA_ROOT` in `local.env` to the directory containing `GS2/` and `GFTM/`. Each arm is a per-code entry of scan information under the same project and case.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
from pyrokinetics import PyroHypercube

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "mtm_default_vs_m11"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
run_template = "Runs"
project = "LATIN_HYPERCUBE"
case = "NSTX_MTM"
scan = "beta_q_shat_ky_n1000"
metadata_file = "pyroscan.json"
output_file = "cube.nc"

reference = ("GS2", f"{scan}/pyro_cube_avg")
arms = {  # panel title -> (code, scan information)
    "GFTM default": ("GFTM", f"{scan}/gftm_fw174/pyro_cube"),
    "GFTM m11_new": ("GFTM", f"{scan}/m11_new/pyro_cube"),
}
gs2_unstable = 1e-3  # c_s/a; GS2 cases at or below this are not scored

## Load data

Pyrokinetics loads each hypercube with its base input and GK output. Rerun only when inputs change.

In [ ]:
def load(code, scan_information):
    directory = data_root / code / run_template / project / case / scan_information
    cube = PyroHypercube(pyroscan_json=directory / metadata_file, load_base_pyro=True)
    cube.from_netcdf(directory / output_file)
    return cube.gk_output.data

gs2 = load(*reference)
data = {title: load(*spec) for title, spec in arms.items()}
print("GS2", dict(gs2.sizes))
for title, ds in data.items():
    print(title, dict(ds.sizes))

## Calculate and select

Arms are reindexed onto the GS2 `sample_name` order (a missing case raises). The dominant mode is the `argmax` of `growth_rate` over `mode`; rows with no finite mode become NaN and drop out of the metrics. The paired mask keeps GS2-unstable cases where every arm has $\gamma > 0$. The count of selected modes that are not index 0 is printed: argmax and index 0 coincide only if it is 0.

In [ ]:
names = [str(n) for n in gs2.sample_name.values]
gamma_gs2 = np.asarray(gs2.growth_rate.values, float)

gamma, nonzero = {}, {}
for title, ds in data.items():
    ds = ds.isel(sample=[list(map(str, ds.sample_name.values)).index(n) for n in names])
    g = np.asarray(ds.growth_rate.transpose("sample", "mode").values, float)  # plain c_s/a numbers
    g = np.where(np.isfinite(g), g, -np.inf)
    k = g.argmax(axis=1)
    gamma[title] = np.where(np.isfinite(g.max(axis=1)), g.max(axis=1), np.nan)
    nonzero[title] = k

paired = (gamma_gs2 > gs2_unstable) & np.all([g > 0 for g in gamma.values()], axis=0)
print(f"GS2-unstable: {(gamma_gs2 > gs2_unstable).sum()}, paired: {paired.sum()}")
print({t: int((nonzero[t][paired] != 0).sum()) for t in nonzero}, "= selected mode not index 0")

stats = {}
for title, g in gamma.items():
    x, y = gamma_gs2[paired], g[paired]
    ok = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x, y = x[ok], y[ok]
    stats[title] = dict(n=len(x), rmse=np.sqrt(np.mean((y - x) ** 2)), bias=np.mean(y - x),
                        r=np.corrcoef(np.log10(x), np.log10(y))[0, 1])
    print(title, stats[title])

## Plot

One log-log parity panel per arm, GS2 on the x axis with $y=x$ dashed, on shared axes limits.

In [ ]:
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(10, 5))
lim = (1e-3, 2 * max(np.nanmax(gamma_gs2[paired]), *(np.nanmax(g[paired]) for g in gamma.values())))
for ax, (title, g) in zip(axes, gamma.items()):
    s = stats[title]
    ax.scatter(gamma_gs2[paired], g[paired], s=8, alpha=0.5, marker="o")
    ax.plot(lim, lim, "k--", marker="")
    ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim, aspect="equal",
           xlabel=r"GS2 $\gamma\,a/c_s$", title=f"{title} (n={s['n']})")
    ax.text(0.04, 0.96, f"RMSE {s['rmse']:.4f}\nbias {s['bias']:+.4f}\nr {s['r']:.4f}",
            transform=ax.transAxes, va="top")
axes[0].set_ylabel(r"GFTM $\gamma\,a/c_s$")
plt.show()

## Save

Replaces the same filename.

In [ ]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / f"{scan}_paired.png")

## Interpretation

Compare the printed metrics with the reference table `results/nstx_mtm_lhc_default_vs_m11/csv/summary_default_vs_m11.csv` (n1000, paired): default RMSE 0.2790 / bias +0.0877 / r 0.5754, m11_new 0.3093 / +0.1272 / 0.5945, n = 833.

Both arms are compared on the same cases, so differences between panels are not a population effect. m11_new correlates slightly better (higher $r$) but overpredicts more (larger positive bias and RMSE).

**Not reproduced here:** the earlier figure distinguished MTM from not-an-MTM points and reported tearing-parity fractions. Those come from the `MTM_analysis` mode detector, which this notebook deliberately does not use, so no such marker or fraction appears. Parity-based claims need that separate analysis.